# 🚴 M2_01 Solutions: Amsterdam Bike Data Acquisition

**Complete solutions with explanations**

---

## 📋 How to Use This Notebook

1. **Try first!** Attempt each task in the main notebook before checking solutions
2. **Learn from differences**: Compare your approach with the solution
3. **Understand, don't copy**: Read the explanations and comments

---

## Setup (Same as Main Notebook)

In [ ]:
# Standard library
import os
import sys
import json
from datetime import datetime
from pathlib import Path

# Third-party imports
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas display
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Setup complete!")

## Configuration

In [ ]:
# API Configuration
BASE_URL = "http://api.citybik.es/v2"
NETWORK_ID = "ov-fiets"
NETWORK_URL = f"{BASE_URL}/networks/{NETWORK_ID}"

# File paths
DATA_DIR = Path('../../data/raw') if 'notebooks' in os.getcwd() else Path('data/raw')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Output filename with timestamp
TIMESTAMP = datetime.now().strftime('%Y-%m-%d_%H%M%S')
OUTPUT_FILE = DATA_DIR / f'amsterdam_bike_{TIMESTAMP}.json'

print("✅ Configuration set")
print(f"🌐 API URL: {NETWORK_URL}")
print(f"💾 Output file: {OUTPUT_FILE}")

---

## ✅ Task 4.1 Solution: Make Your First API Request

**Note:** This corresponds to Part 4 in the main notebook.

In [ ]:
# SOLUTION: Task 4.1 - Fetch bike data

print("📡 Fetching bike data from CityBikes API...")
print(f"🔗 URL: {NETWORK_URL}\n")

# Make GET request
response = requests.get(NETWORK_URL)

# Check if successful
if response.status_code == 200:
    # Parse JSON
    data = response.json()
    
    # Extract information
    network_name = data['network']['name']
    city = data['network']['location']['city']
    country = data['network']['location']['country']
    num_stations = len(data['network']['stations'])
    
    # Print results
    print(f"✅ Success! Fetched data for:")
    print(f"   Network: {network_name}")
    print(f"   Location: {city}, {country}")
    print(f"   Stations: {num_stations}")
else:
    print(f"❌ Failed: HTTP {response.status_code}")

---

## ✅ Task 4.2 Solution: Explore the JSON Structure

**Note:** This corresponds to Part 4 in the main notebook.

In [ ]:
# SOLUTION: Task 4.2 - Explore JSON structure

# Get first station
first_station = data['network']['stations'][0]

# Pretty print with json.dumps
print("📋 First station structure:")
print(json.dumps(first_station, indent=2))

print("\n" + "="*60)
print("Field types:")
print("="*60)
for key, value in first_station.items():
    print(f"{key:20s} : {type(value).__name__}")

---

## ✅ Task 5.1 Solution: Implement Robust Error Handling

**Note:** This corresponds to Part 5 in the main notebook.

In [ ]:
# SOLUTION: Task 5.1 - Comprehensive error handling

def fetch_bike_data_safe(network_id, timeout=10):
    """
    Fetch bike data with comprehensive error handling.
    
    Parameters:
    -----------
    network_id : str
        The network ID (e.g., 'ov-fiets')
    timeout : int
        Request timeout in seconds (default: 10)
    
    Returns:
    --------
    dict or None
        JSON data if successful, None if any error occurs
    """
    url = f"{BASE_URL}/networks/{network_id}"
    
    try:
        # Make request with timeout
        print(f"📡 Fetching data from: {url}")
        response = requests.get(url, timeout=timeout)
        
        # Raise exception for bad HTTP status (4xx, 5xx)
        response.raise_for_status()
        
        # Parse and return JSON
        data = response.json()
        print(f"✅ Success! Fetched {len(data['network']['stations'])} stations")
        return data
        
    except requests.exceptions.Timeout:
        print(f"⏱️ Timeout: Request took longer than {timeout} seconds")
        return None
        
    except requests.exceptions.ConnectionError:
        print("🔌 Connection Error: Could not connect to API")
        return None
        
    except requests.exceptions.HTTPError as e:
        print(f"❌ HTTP Error {e.response.status_code}: {e}")
        if e.response.status_code == 404:
            print(f"   Network '{network_id}' not found")
        return None
        
    except requests.exceptions.RequestException as e:
        print(f"⚠️ Request Error: {e}")
        return None
        
    except json.JSONDecodeError:
        print("📝 JSON Error: Could not parse response")
        return None


# Test the function
print("🧪 Testing error handling function...")
print("=" * 60)

# Test 1: Valid network
print("\nTest 1: Valid network (ov-fiets)")
result = fetch_bike_data_safe(NETWORK_ID, timeout=10)
if result:
    print(f"✅ Test 1 passed!\n")

# Test 2: Invalid network
print("\nTest 2: Invalid network ID")
result = fetch_bike_data_safe("fake-network-12345", timeout=10)
if result is None:
    print("✅ Test 2 passed\n")

print("=" * 60)

---

## 📊 Part 7: Convert to DataFrame - Solutions

**Note:** These solutions correspond to Part 7 in the main notebook.

**Important:** The solution names the DataFrame `df_bikes` - this variable is used throughout.

In [ ]:
# Fetch data using our safe function
data = fetch_bike_data_safe(NETWORK_ID)
if not data:
    raise ValueError("Failed to fetch data")

### Task 7.1 Solution: Extract and Convert to DataFrame

In [ ]:
# SOLUTION: Task 7.1 - Extract and convert to DataFrame (named df_bikes)

# Extract stations list
stations = data['network']['stations']

# Convert to DataFrame
df_bikes = pd.DataFrame(stations)

# Display info
print(f"✅ Created DataFrame: {df_bikes.shape[0]} rows × {df_bikes.shape[1]} columns\n")
print("First 3 rows:")
display(df_bikes.head(3))

print("\n" + "="*60)
print("DataFrame Info:")
print("="*60)
df_bikes.info()

### Task 7.2 Solution: Clean Column Names and Parse Timestamps

In [ ]:
# SOLUTION: Task 7.2 - Clean column names and parse timestamps

# Parse timestamp
df_bikes['timestamp'] = pd.to_datetime(df_bikes['timestamp'])

# Rename columns
df_bikes = df_bikes.rename(columns={
    'free_bikes': 'bikes_available',
    'empty_slots': 'docks_available'
})

print("✅ Parsed timestamps and renamed columns")
print(f"\nTimestamp column type: {df_bikes['timestamp'].dtype}")

### Task 7.3 Solution: Add Derived Columns

In [ ]:
# SOLUTION: Task 7.3 - Add derived columns

# Total capacity
df_bikes['total_capacity'] = df_bikes['bikes_available'] + df_bikes['docks_available']

# Utilization percentage
df_bikes['utilization_pct'] = (
    df_bikes['bikes_available'] / df_bikes['total_capacity'] * 100
).round(2)

# Empty flag
df_bikes['is_empty'] = df_bikes['bikes_available'] == 0

print("✅ Added derived columns")
print("\nSample values:")
display(df_bikes[['bikes_available', 'docks_available', 'total_capacity', 
                   'utilization_pct', 'is_empty']].head())

### Task 7.4 Solution: Add Network Metadata

In [ ]:
# SOLUTION: Task 7.4 - Add network metadata

df_bikes['network_id'] = data['network']['id']
df_bikes['network_name'] = data['network']['name']
df_bikes['city'] = data['network']['location']['city']
df_bikes['country'] = data['network']['location']['country']

print("✅ Added network metadata")
print(f"\nNetwork: {df_bikes['network_name'].iloc[0]}")
print(f"Location: {df_bikes['city'].iloc[0]}, {df_bikes['country'].iloc[0]}")

### Task 7.5 Solution: Reorder and Validate

In [ ]:
# SOLUTION: Task 7.5 - Reorder columns and validate

# Define column order
column_order = [
    'id', 'name', 'latitude', 'longitude',
    'bikes_available', 'docks_available', 'total_capacity', 'utilization_pct',
    'timestamp', 'network_id', 'network_name', 'city', 'country'
]

# Reorder
df_bikes = df_bikes[column_order]

print("✅ Reordered columns\n")
print("First 10 rows:")
display(df_bikes.head(10))

print("\n" + "="*60)
print("Summary Statistics:")
print("="*60)
display(df_bikes[['bikes_available', 'docks_available', 
                   'total_capacity', 'utilization_pct']].describe())

---

## 🔍 Part 8: Data Quality and Exploration - Solutions

**Note:** These solutions correspond to Part 8 in the main notebook.

### Task 8.1 Solution: Calculate Summary Statistics

In [ ]:
# SOLUTION: Task 8.1 - Calculate summary statistics

print("📈 Summary Statistics:")
print("="*60)
display(df_bikes[['bikes_available', 'docks_available', 
                   'total_capacity', 'utilization_pct']].describe())

print("\n" + "="*60)
print("🚴 Bike Availability Summary:")
print("="*60)
print(f"Total Stations: {len(df_bikes)}")
print(f"Total Bikes Available: {df_bikes['bikes_available'].sum()}")
print(f"Total Docks Available: {df_bikes['docks_available'].sum()}")
print(f"Average Station Capacity: {df_bikes['total_capacity'].mean():.1f}")
print(f"Average Utilization: {df_bikes['utilization_pct'].mean():.1f}%")

print("\n" + "="*60)
print("⚠️ Problem Stations:")
print("="*60)
empty_stations = df_bikes[df_bikes['bikes_available'] == 0]
full_stations = df_bikes[df_bikes['docks_available'] == 0]
print(f"Empty stations (no bikes): {len(empty_stations)}")
print(f"Full stations (no docks): {len(full_stations)}")

### Task 8.2 Solution: Create 4-Panel Visualization Dashboard

In [ ]:
# SOLUTION: Task 8.2 - Create 4-panel visualization dashboard

# Create 2x2 subplot figure
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('📊 Bike Availability Dashboard', fontsize=16, fontweight='bold')

# Panel 1: Histogram of bikes_available
axes[0, 0].hist(df_bikes['bikes_available'], bins=20, color='steelblue', 
                edgecolor='black', alpha=0.7)
mean_bikes = df_bikes['bikes_available'].mean()
axes[0, 0].axvline(mean_bikes, color='red', linestyle='--', linewidth=2, 
                    label=f'Mean: {mean_bikes:.1f}')
axes[0, 0].set_xlabel('Bikes Available')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Available Bikes')
axes[0, 0].legend()
axes[0, 0].grid(axis='y', alpha=0.3)

# Panel 2: Histogram of utilization_pct
axes[0, 1].hist(df_bikes['utilization_pct'], bins=20, color='darkorange', 
                edgecolor='black', alpha=0.7)
mean_util = df_bikes['utilization_pct'].mean()
axes[0, 1].axvline(mean_util, color='red', linestyle='--', linewidth=2,
                    label=f'Mean: {mean_util:.1f}%')
axes[0, 1].set_xlabel('Utilization (%)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of Station Utilization')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# Panel 3: Top 10 stations by bikes_available
top_10_bikes = df_bikes.nlargest(10, 'bikes_available')
axes[1, 0].barh(range(len(top_10_bikes)), top_10_bikes['bikes_available'], 
                color='seagreen', edgecolor='black')
axes[1, 0].set_yticks(range(len(top_10_bikes)))
axes[1, 0].set_yticklabels(top_10_bikes['name'], fontsize=9)
axes[1, 0].set_xlabel('Bikes Available')
axes[1, 0].set_title('Top 10 Stations by Available Bikes')
axes[1, 0].invert_yaxis()  # Highest at top
axes[1, 0].grid(axis='x', alpha=0.3)

# Panel 4: Average bikes vs docks
avg_bikes = df_bikes['bikes_available'].mean()
avg_docks = df_bikes['docks_available'].mean()
bars = axes[1, 1].bar(['Bikes Available', 'Docks Available'], 
                       [avg_bikes, avg_docks],
                       color=['steelblue', 'coral'], edgecolor='black')
axes[1, 1].set_ylabel('Average Count')
axes[1, 1].set_title('Average Station Availability')
axes[1, 1].grid(axis='y', alpha=0.3)

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.1f}',
                    ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ 4-panel dashboard created!")

### Task 8.3 Solution: Create Geographic Station Map

In [ ]:
# SOLUTION: Task 8.3 - Create geographic station map

# Create figure
plt.figure(figsize=(12, 8))

# Create scatter plot with color and size mapping
scatter = plt.scatter(
    df_bikes['longitude'], 
    df_bikes['latitude'],
    c=df_bikes['bikes_available'],  # Color by bikes available
    s=df_bikes['total_capacity'] * 5,  # Size by capacity
    cmap='RdYlGn',  # Red=low, green=high
    edgecolors='black',
    linewidth=0.5,
    alpha=0.7
)

# Add colorbar
cbar = plt.colorbar(scatter, label='Bikes Available')

# Find and annotate largest station
largest_station = df_bikes.nlargest(1, 'total_capacity').iloc[0]
plt.annotate(
    f"{largest_station['name']}\n(Capacity: {largest_station['total_capacity']})",
    xy=(largest_station['longitude'], largest_station['latitude']),
    xytext=(10, 10),
    textcoords='offset points',
    bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7),
    arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0', color='black'),
    fontsize=9,
    fontweight='bold'
)

# Formatting
plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.title('🗺️ Geographic Distribution of Bike Stations\n(Size = Capacity, Color = Bikes Available)', 
          fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Geographic map created!")

### Task 8.4 Solution: Custom Analysis (Example)

**Example Analysis:** Station Utilization Categories

This example shows how to categorize stations and visualize the distribution of problematic vs healthy stations.

In [ ]:
# SOLUTION: Task 8.4 - Custom analysis example
# Analysis: Station Utilization Categories

# Categorize stations
def categorize_station(row):
    """Categorize station based on availability."""
    if row['bikes_available'] == 0:
        return 'Empty (No Bikes)'
    elif row['docks_available'] == 0:
        return 'Full (No Docks)'
    elif row['utilization_pct'] < 25:
        return 'Low Utilization (<25%)'
    elif row['utilization_pct'] > 75:
        return 'High Utilization (>75%)'
    else:
        return 'Healthy (25-75%)'

df_bikes['station_status'] = df_bikes.apply(categorize_station, axis=1)

# Count by category
category_counts = df_bikes['station_status'].value_counts()

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Pie chart
colors = ['#ff6b6b', '#ffd93d', '#95e1d3', '#6bcf7f', '#4d96ff']
ax1.pie(category_counts.values, labels=category_counts.index, autopct='%1.1f%%',
        colors=colors, startangle=90, textprops={'fontsize': 10})
ax1.set_title('Station Status Distribution', fontsize=14, fontweight='bold')

# Bar chart with counts
bars = ax2.bar(range(len(category_counts)), category_counts.values, 
               color=colors, edgecolor='black')
ax2.set_xticks(range(len(category_counts)))
ax2.set_xticklabels(category_counts.index, rotation=45, ha='right', fontsize=10)
ax2.set_ylabel('Number of Stations')
ax2.set_title('Station Count by Status', fontsize=14, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Add count labels on bars
for bar in bars:
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{int(height)}',
             ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print("\\n📊 Station Status Summary:")
print("=" * 60)
for status, count in category_counts.items():
    pct = (count / len(df_bikes)) * 100
    print(f"{status:30s}: {count:3d} stations ({pct:5.1f}%)")
print("=" * 60)

# Calculate percentage of problematic stations
problematic = category_counts.get('Empty (No Bikes)', 0) + category_counts.get('Full (No Docks)', 0)
problematic_pct = (problematic / len(df_bikes)) * 100

print(f"\\n⚠️ Problematic stations (empty or full): {problematic} ({problematic_pct:.1f}%)")
print(f"✅ Healthy stations: {len(df_bikes) - problematic} ({100-problematic_pct:.1f}%)")

print("\\n✅ Custom analysis complete!")

---

## 📚 Key Takeaways

### From Task 4.1-4.2 (API Basics - Part 4)
- ✅ APIs return structured data (usually JSON)
- ✅ Always check status codes (200 = success)
- ✅ Explore JSON structure before DataFrame conversion

### From Task 5.1 (Error Handling - Part 5)
- ✅ Use specific exception types
- ✅ Order exceptions from specific to general
- ✅ Return None for easy error checking
- ✅ Always include timeout parameter

### From Tasks 7.1-7.5 (DataFrame Operations - Part 7)
- ✅ pd.DataFrame() directly converts list of dicts
- ✅ Parse timestamps with pd.to_datetime()
- ✅ Vectorized operations are fast and clean
- ✅ Derived columns add analytical value

### From Tasks 8.1-8.4 (Visualization - Part 8)
- ✅ Start with summary statistics (.describe())
- ✅ Identify problem areas (empty/full stations)
- ✅ Multiple visualizations tell complete story

---

**🎉 Congratulations! You now have production-ready API data acquisition skills!**

**Next Steps:**
- M2_02: Weather Data API
- M2_03: Data Storage Best Practices
- M2_04: Merge Datasets